# Improve the Model

In the previous page, we trained a baseline XGBoost regressor on the Australian rental market dataset. Our tuned baseline reached roughly:

- **MAE ≈ \$130 / week**
- **RMSE ≈ \$270 / week**
- **R² ≈ 0.4**

That is a useful starting point but not yet production-grade. In this page we walk through four practical levers that real pricing teams use to lift model quality:

1. **Target transformation** — handling the right-skewed rent distribution with a log transform.
2. **Feature engineering** — extracting signal from `amenities`, `description`, and geographic data.
3. **Hyperparameter tuning** — running a randomized search instead of guessing one configuration.
4. **Integrating additional data** — ideas for enriching the dataset with external sources.

:::{important} Goal
Each technique is introduced on its own and then we layer them together at the end to compare against the baseline. The point is not to chase a perfect score, but to show _how_ an analyst iterates on a model.
:::


In [1]:
import numpy as np
import pandas as pd

import sklearn
from sklearn.model_selection import train_test_split, KFold, RandomizedSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.feature_extraction.text import TfidfVectorizer

import xgboost
from xgboost import XGBRegressor

import plotly.express as px

from scipy.stats import randint, uniform

print(f"pandas version:       {pd.__version__}")
print(f"scikit-learn version: {sklearn.__version__}")
print(f"xgboost version:      {xgboost.__version__}")


pandas version:       2.3.3
scikit-learn version: 1.7.2
xgboost version:      3.1.3


## Load Data


In [2]:
df = pd.read_csv(
    "https://github.com/bdi593/datasets/raw/refs/heads/main/australian-rental-market-2026/australian_rental_market_2026.csv"
)
df.shape


(6767, 16)

We will hold out the same 20% test set we used in the baseline notebook so that the comparison is apples-to-apples.


In [3]:
target = "price_display"
y = df[target]

df_train, df_test, y_train, y_test = train_test_split(
    df, y, test_size=0.2, random_state=42
)

print("Train:", df_train.shape, "Test:", df_test.shape)


Train: (5413, 16) Test: (1354, 16)


## 1. Target Transformation

The histogram in the baseline notebook showed that `price_display` is strongly right-skewed: a small number of luxury listings stretch into the \$3,000–\$5,000+ per week range. When the target is skewed, squared-error loss becomes dominated by these extreme observations and the model effectively over-fits to a handful of rows.

A standard remedy is to model `log(1 + price)` instead of `price`. Predictions are then exponentiated back to dollars before evaluation.

:::{tip} Why `log1p` and not `log`?
`np.log1p(x)` computes `log(1 + x)`, which is well-defined when `x = 0` and is numerically more stable than `log(x)` for small positive values. The inverse is `np.expm1(y) = exp(y) - 1`.
:::


In [4]:
y_train_log = np.log1p(y_train)
y_test_log = np.log1p(y_test)

fig = px.histogram(
    x=y_train_log,
    nbins=50,
    title="log(1 + weekly rent) — much closer to symmetric",
    template="simple_white",
)
fig.update_layout(xaxis_title="log1p(price_display)", yaxis_title="count")
fig.show()


## 2. Feature Engineering

The baseline only used the structured columns. Several of the most informative fields were ignored. We will add three groups of features:

1. **Amenity flags** parsed from the comma-separated `amenities` text.
2. **Description length and TF–IDF features** from the `description` column.
3. **Geographic aggregates** — suburb-level median rent and rough distances to the five largest Australian capital cities.


### 2a. Amenity flags

The `amenities` column is a free-text list such as `"Air Conditioning, Balcony, Dishwasher"`. Splitting it gives us a small set of boolean features that are easy to interpret.


In [5]:
AMENITY_KEYWORDS = [
    "air conditioning",
    "balcony",
    "dishwasher",
    "pool",
    "gym",
    "garage",
    "furnished",
    "heating",
    "built in robes",
    "study",
]


def add_amenity_flags(frame: pd.DataFrame) -> pd.DataFrame:
    text = frame["amenities"].fillna("").str.lower()
    out = frame.copy()
    for kw in AMENITY_KEYWORDS:
        col = "amen_" + kw.replace(" ", "_")
        out[col] = text.str.contains(kw, regex=False).astype(int)
    return out


df_train_fe = add_amenity_flags(df_train)
df_test_fe = add_amenity_flags(df_test)

amenity_cols = ["amen_" + kw.replace(" ", "_") for kw in AMENITY_KEYWORDS]
df_train_fe[amenity_cols].mean().round(3).to_frame("share_of_listings")


,share_of_listings
amen_air_conditioning,0.408
amen_balcony,0.143
amen_dishwasher,0.255
amen_pool,0.045
amen_gym,0.028
amen_garage,0.563
amen_furnished,0.052
amen_heating,0.128
amen_built_in_robes,0.000
amen_study,0.044


### 2b. Description-based features

Listings with longer, richer descriptions tend to be either premium properties (more to brag about) or carefully marketed rentals. We add the raw description length as a feature and a small set of TF–IDF features that capture words like _luxury_, _renovated_, _waterfront_, _new_, etc.

:::{seealso} Why limit `max_features`?
TF–IDF on tens of thousands of unique tokens would explode the feature space and hurt XGBoost's training speed. Capping `max_features` at a few hundred keeps the model nimble while preserving the highest-information words.
:::


In [6]:
df_train_fe["desc_length"] = df_train_fe["description"].fillna("").str.len()
df_test_fe["desc_length"] = df_test_fe["description"].fillna("").str.len()

tfidf = TfidfVectorizer(
    max_features=200,
    stop_words="english",
    ngram_range=(1, 1),
    min_df=20,
)
X_text_train = tfidf.fit_transform(df_train_fe["description"].fillna(""))
X_text_test = tfidf.transform(df_test_fe["description"].fillna(""))

tfidf_cols = [f"tfidf_{w}" for w in tfidf.get_feature_names_out()]
X_text_train_df = pd.DataFrame(
    X_text_train.toarray(), columns=tfidf_cols, index=df_train_fe.index
)
X_text_test_df = pd.DataFrame(
    X_text_test.toarray(), columns=tfidf_cols, index=df_test_fe.index
)

print(f"TF-IDF feature count: {len(tfidf_cols)}")


TF-IDF feature count: 200


### 2c. Geographic feature engineering

Latitude and longitude are useful, but trees split on them axis-by-axis. Two additions usually help:

1. **Distance to the nearest major city.** A rough Haversine distance to each of Sydney, Melbourne, Brisbane, Perth, and Adelaide.
2. **Suburb-level price summaries.** The median rent of comparable listings in the same suburb is a strong prior. We compute it on the training set only to avoid target leakage.

:::{caution} Avoiding target leakage
We compute the suburb median on `df_train_fe` (with the training target) and then _join_ that lookup table onto both train and test sets. We never let test-set rents influence training features.
:::


In [7]:
CAPITALS = {
    "sydney": (-33.8688, 151.2093),
    "melbourne": (-37.8136, 144.9631),
    "brisbane": (-27.4698, 153.0251),
    "perth": (-31.9523, 115.8613),
    "adelaide": (-34.9285, 138.6007),
}


def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    return 6371.0 * 2 * np.arcsin(np.sqrt(a))


def add_geo_features(frame: pd.DataFrame) -> pd.DataFrame:
    out = frame.copy()
    for city, (clat, clon) in CAPITALS.items():
        out[f"dist_km_{city}"] = haversine_km(
            out["latitude"], out["longitude"], clat, clon
        )
    out["dist_km_nearest_capital"] = out[[f"dist_km_{c}" for c in CAPITALS]].min(axis=1)
    return out


df_train_fe = add_geo_features(df_train_fe)
df_test_fe = add_geo_features(df_test_fe)


In [8]:
# Suburb-level median rent computed on TRAIN only to avoid leakage
suburb_stats = (
    df_train_fe.groupby("suburb")["price_display"].median().rename("suburb_median_rent")
)
global_median = df_train_fe["price_display"].median()

df_train_fe = df_train_fe.join(suburb_stats, on="suburb")
df_test_fe = df_test_fe.join(suburb_stats, on="suburb")

df_train_fe["suburb_median_rent"] = df_train_fe["suburb_median_rent"].fillna(
    global_median
)
df_test_fe["suburb_median_rent"] = df_test_fe["suburb_median_rent"].fillna(
    global_median
)


### Assemble the engineered feature matrix


In [9]:
numeric_features = (
    [
        "bedrooms",
        "bathrooms",
        "parking_spaces",
        "latitude",
        "longitude",
        "desc_length",
        "suburb_median_rent",
        "dist_km_nearest_capital",
    ]
    + [f"dist_km_{c}" for c in CAPITALS]
    + amenity_cols
)

categorical_features = [
    "propertyType",
    "locality",
    "postcode",
    "state",
    "suburb",
    "agency_name",
]

for col in categorical_features:
    df_train_fe[col] = df_train_fe[col].astype("category")
    df_test_fe[col] = df_test_fe[col].astype("category")

X_train = pd.concat(
    [df_train_fe[numeric_features + categorical_features], X_text_train_df], axis=1
)
X_test = pd.concat(
    [df_test_fe[numeric_features + categorical_features], X_text_test_df], axis=1
)

print("Engineered feature count:", X_train.shape[1])


Engineered feature count: 229


## 3. Hyperparameter Tuning with `RandomizedSearchCV`

In the baseline notebook we picked one set of hyperparameters by hand. A more rigorous approach is to _search_ over a distribution of plausible values and let cross-validation pick the best combination.

We use `RandomizedSearchCV` rather than `GridSearchCV` because it gives us much more coverage of the search space for the same number of fits. We optimize on the **log-transformed target** so the search is robust to skew.

:::{tip} Random search vs grid search
A 4-dimensional grid with 5 values per axis is `5⁴ = 625` configurations. A random search of 30 draws from continuous distributions typically finds a near-best configuration in a tiny fraction of the time.
:::


:::{warning} Long-running code ahead

The code below will take a while to run, especially if you have a large dataset or a complex model. You do not need to run it to understand the concept, but feel free to experiment with it if you have the resources.

:::


In [10]:
param_dist = {
    "n_estimators": randint(400, 1500),
    "max_depth": randint(4, 10),
    "learning_rate": uniform(0.02, 0.15),
    "subsample": uniform(0.6, 0.4),
    "colsample_bytree": uniform(0.6, 0.4),
    "reg_lambda": uniform(0.5, 4.0),
    "min_child_weight": randint(1, 10),
}

search = RandomizedSearchCV(
    estimator=XGBRegressor(
        objective="reg:squarederror",
        tree_method="hist",
        enable_categorical=True,
        random_state=42,
        n_jobs=-1,
    ),
    param_distributions=param_dist,
    n_iter=20,
    scoring="neg_mean_absolute_error",
    cv=KFold(n_splits=3, shuffle=True, random_state=42),
    verbose=1,
    random_state=42,
    n_jobs=1,  # XGBoost itself is parallel; nesting can over-subscribe CPUs
)

search.fit(X_train, np.log1p(y_train))
search.best_params_


Fitting 3 folds for each of 20 candidates, totalling 60 fits


{'colsample_bytree': np.float64(0.7218455076693483),
 'learning_rate': np.float64(0.034650817100957576),
 'max_depth': 7,
 'min_child_weight': 7,
 'n_estimators': 908,
 'reg_lambda': np.float64(3.8327796469446573),
 'subsample': np.float64(0.6693458614031088)}

:::{tip} Avoiding nested parallelism

When using `RandomizedSearchCV` with models like XGBoost, it's important to avoid nested parallelism, where both the model and the search process try to use all CPU cores at the same time. XGBoost already parallelizes training internally (via `n_jobs=-1`), so setting `n_jobs=1` in `RandomizedSearchCV` ensures that only one model is trained at a time, preventing CPU oversubscription. If both levels were parallelized, multiple models would compete for the same cores, leading to slower performance due to overhead and resource contention. The general rule is to parallelize at only one level - either within the model or across the hyperparameter search - not both.

:::


## Evaluate the Improved Model

We refit the model on the full training set with the best hyperparameters, predict on the held-out test set, and exponentiate predictions back to dollars before computing the metrics. This is the only fair way to compare against the baseline.


In [11]:
best_model = search.best_estimator_

y_pred_log = best_model.predict(X_test)
y_pred = np.expm1(y_pred_log)

mae = mean_absolute_error(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred) ** 0.5
r2 = r2_score(y_test, y_pred)

print(f"Improved MAE:  ${mae:,.2f} AUD/week")
print(f"Improved RMSE: ${rmse:,.2f} AUD/week")
print(f"Improved R²:   {r2:.3f}")


Improved MAE:  $97.00 AUD/week
Improved RMSE: $203.97 AUD/week
Improved R²:   0.658


### Inspect feature importance

Looking at which features the model relies on is a quick sanity check. We expect the new `suburb_median_rent`, the distance-to-capital features, and the structured property attributes to dominate.


In [14]:
importances = (
    pd.Series(best_model.feature_importances_, index=X_train.columns)
    .sort_values(ascending=False)
    .head(20)
)

fig = px.bar(
    importances.iloc[::-1],
    orientation="h",
    title="Top 20 features by XGBoost gain",
    labels={"value": "Importance", "index": "Feature"},
    height=600,
    template="simple_white",
)
fig.update_layout(showlegend=False)
fig.show()


## 4. Integrating Additional Data

Even with careful feature engineering and tuning, we are still limited by what is _in the dataset_. A real pricing engine would enrich each listing with external context.

Useful sources include:

- **Census / ABS data** at the SA2 or postcode level: median household income, age distribution, household composition, employment rate.
- **School catchment quality** — ICSEA scores, NAPLAN performance.
- **Transit accessibility** — walking distance to the nearest train/tram station, commute time to the CBD via Google Distance Matrix or OSRM.
- **Points of interest** — number of cafés, supermarkets, parks within 1 km.
- **Macro indicators** — RBA cash rate, CoreLogic price index, vacancy rates.
- **Listing dynamics** — days on market, view counts, number of price drops.

### Explanations of the above features:

**SA2 (Statistical Area Level 2)**\
A geographic unit defined by the Australian Bureau of Statistics (ABS). It represents a medium-sized area (typically a suburb or group of suburbs) designed to reflect communities with similar social and economic characteristics. Using SA2-level data lets you attach neighborhood-level features like median income or employment rates to each listing.

**ABS (Australian Bureau of Statistics)**

Australia's official statistics agency. It provides census data and other demographic/economic indicators (income, age distribution, etc.), often aggregated at levels like SA2 or postcode.

**ICSEA (Index of Community Socio-Educational Advantage)**

A score assigned to schools that reflects the socio-educational background of their students (based on factors like parental income, education, and occupation). Higher ICSEA generally indicates a more advantaged student population, which is often correlated with perceived school quality and can influence property prices.

**NAPLAN (National Assessment Program - Literacy and Numeracy)**

A standardized test taken by Australian students in Years 3, 5, 7, and 9. School-level NAPLAN results are commonly used as a proxy for academic performance, and thus another signal of school quality for nearby properties.

**RBA Cash Rate**

The benchmark interest rate set by the Reserve Bank of Australia (RBA). It influences mortgage rates and borrowing costs, making it a key macroeconomic driver of housing demand and prices.

**CoreLogic Price Index**

A widely used property market index that tracks housing price trends over time across regions. It provides a sense of broader market movement (e.g., whether prices are rising or falling overall).

**Vacancy Rates**

The percentage of rental properties that are unoccupied in a given area. Lower vacancy rates typically indicate strong rental demand, which can push prices up.

**Transit and POI metrics (not acronyms but important)**

- _Transit accessibility_: Measures like distance to the nearest station or commute time to the city center (CBD), often computed via APIs (e.g., Google Distance Matrix).
- _Points of interest (POIs)_: Counts of nearby amenities (cafés, parks, supermarkets), which help quantify neighborhood livability.

:::{hint} Joining external data
Most of these sources are available at the postcode or SA2 level. Once you have a lookup table keyed by postcode you can simply join it onto our DataFrame the same way we joined `suburb_median_rent`.
:::

Recall that this is an Australian rental market dataset, so the external data would be Australian-specific, but the general principle applies everywhere: the more you can tell about a property and its surroundings, the better the model can learn patterns in how those factors influence price.


## Summary

| Iteration               | What changed                                                | Typical impact                |
| ----------------------- | ----------------------------------------------------------- | ----------------------------- |
| Baseline                | Default hyperparameters, 5 numeric + 6 categorical features | MAE ≈ \$130, R² ≈ 0.4         |
| + log target            | Stabilizes loss against right-skewed prices                 | Lower RMSE on luxury listings |
| + amenity flags         | Adds 10 boolean signals from a free-text column             | Small but consistent lift     |
| + TF–IDF on description | Captures "luxury", "new", "waterfront" cues                 | Helps premium segments        |
| + geographic features   | Distance to capitals + suburb median rent                   | Often the largest single jump |
| + tuned hyperparameters | `RandomizedSearchCV` over 20 configurations                 | Final 5–15% improvement       |

The exact gains depend on the random seed and the search budget, but the pattern is consistent: most of the lift comes from **better features**, not from squeezing the last drop out of the model. When you next iterate on a regressor, prioritize your time accordingly.
